![IITIS](pictures/logoIITISduze.png)

# Algorytmu wyczerpującego przeszukiwania

Inna nazwa to brute force. Tutaj użyjemy potężniejszych wersji tego algorytmu. Ogólna ide jest 


In [ ]:
# implementacja algorytmu


def brute_force_naive(J, h):
    ...



def brute_force_gpu():
    ...

## Basics of parrarel procesings

In [2]:
import numpy as np

from itertools import product
from multiprocessing import Pool, cpu_count
from tqdm import tqdm
from math import inf


# Funkcje pomocniczne



def calculate_energy(J: np.ndarray, h: np.ndarray, state: np.ndarray):
    return state @ J @ state.T + state @ h 


def process_state(J: np.ndarray, h: np.ndarray, state: list):
    state = np.array(state)
    en = calculate_energy(J, h, state)
    return en, state

def process_states_wrapper(args):
    return process_state(*args)


def brute_force_multithread(J, h, num_workers: int = None, batch_size: int = 1024):
    n = len(h)

    if num_workers is None:
        num_workers = cpu_count()

    n = len(h)
    all_states = list(product([-1, 1], repeat=n))
    args = [(J, h, state) for state in all_states]
    
    with Pool(num_workers) as pool:
        
        results = list(
            tqdm(pool.imap_unordered(process_states_wrapper, args),
                 total=len(all_states),
                 desc="Wyczerpujące przeszukiwanie")
        )

    min_energy, min_state = min(results, key=lambda x: x[0])

    return min_energy, min_state

In [ ]:

n = 4

scaling_func = np.vectorize(lambda x: 2*x-1)  # przesunięcie wartości z (0, 1) do (-1, 1)
J = np.triu(scaling_func(np.random.rand(n, n)))  # losowa gęsta macierz górnotrójkątna
h = scaling_func(np.random.rand(n))  # losowy wektor


from dimod import BinaryQuadraticModel
from dwave.samplers import SimulatedAnnealingSampler

bqm_instance = BinaryQuadraticModel(h, J, vartype="SPIN")
sampler= SimulatedAnnealingSampler()
sampleset = sampler.sample(bqm_instance, num_reads=1)
print(sampleset)

s, e = brute_force_multithread(J, h)

print(e)

   0  1  2  3    energy num_oc.
0 +1 +1 -1 +1 -4.357185       1
['SPIN', 1 rows, 1 samples, 4 variables]
all states created
args created
pool created


Wyczerpujące przeszukiwanie:   0%|          | 0/16 [00:00<?, ?it/s]